# AI Studio Control Panel — Google Colab (L4 GPU)

**Canonical source:** This notebook lives in the [GitHub repository](https://github.com/morecolorfulthanHD/ai-studio-colab). Open it from GitHub in Colab — not from a Google Drive copy.

**Repository Sync** (run early) clones or pulls the latest code from GitHub into `/content/ai-studio-colab`.

**Google Drive** is persistent storage only — models, outputs, datasets, references, and checkpoints at:

`/content/drive/MyDrive/AI_Studio`

This notebook is organized as a clean multi-cell control panel for:

- ComfyUI
- Automatic1111
- AnimateDiff
- Stable Video Diffusion
- ReActor / FaceSwap / InsightFace
- ControlNet
- ControlNet Aux
- Impact Pack
- WAS Node Suite

Design goals:

- clean startup
- explicit errors
- modular functions
- no silent failures
- fresh runtime installs
- Drive-backed models and settings
- Colab-native port proxy URLs

In [1]:
# ============================================================
# Cell 1 — Environment Checks
# ============================================================

import os
import sys
import json
import time
import shutil
import psutil
import subprocess
from pathlib import Path

def _bytes_to_gb(num_bytes: int) -> float:
    return round(num_bytes / (1024 ** 3), 2)

def require_gpu():
    print("=== Environment Check ===")
    print(f"Python: {sys.version.split()[0]}")
    print(f"Platform: {sys.platform}")

    total_ram = psutil.virtual_memory().total
    print(f"RAM Total: {_bytes_to_gb(total_ram)} GB")

    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            check=True,
            capture_output=True,
            text=True,
        )
        gpus = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    except FileNotFoundError:
        raise SystemExit("ERROR: nvidia-smi not found. This runtime does not appear to have an NVIDIA GPU.")
    except subprocess.CalledProcessError as e:
        raise SystemExit(f"ERROR: Unable to query GPU via nvidia-smi.\n{e.stderr}")

    if not gpus:
        raise SystemExit("ERROR: No NVIDIA GPU detected. Stop now and switch the runtime to L4 GPU.")

    print("GPU(s):")
    for gpu in gpus:
        print(f"  - {gpu}")

    print("\nEnvironment looks good.")

require_gpu()

=== Environment Check ===
Python: 3.12.13
Platform: linux
RAM Total: 52.96 GB
GPU(s):
  - NVIDIA L4, 23034 MiB, 580.82.07

Environment looks good.


In [2]:
# ============================================================
# Cell 2 — Mount Google Drive
# ============================================================

from pathlib import Path

def mount_drive_or_fail():
    try:
        from google.colab import drive
    except Exception as e:
        raise SystemExit(f"ERROR: This notebook must run inside Google Colab.\nDetails: {e}")

    drive_root = Path("/content/drive")
    mydrive = drive_root / "MyDrive"

    print("Mounting Google Drive...")
    try:
        drive.mount(str(drive_root), force_remount=False)
    except Exception as e:
        raise SystemExit(f"ERROR: Google Drive mount failed.\nDetails: {e}")

    if not mydrive.exists():
        raise SystemExit("ERROR: /content/drive/MyDrive does not exist after mount. Stop and remount Drive.")

    print(f"Drive mounted successfully: {drive_root}")
    print(f"MyDrive validated: {mydrive}")

mount_drive_or_fail()

Mounting Google Drive...
Mounted at /content/drive
Drive mounted successfully: /content/drive
MyDrive validated: /content/drive/MyDrive


In [3]:
# ============================================================
# Cell 3 — Define Paths & Settings
# ============================================================

import os
import sys
import json
import time
import hashlib
import tarfile
import shutil
import signal
import threading
import socket
import subprocess
from pathlib import Path
from datetime import datetime

try:
    import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

try:
    from IPython.display import display, HTML
except Exception:
    display = None
    HTML = None

ROOT_DRIVE = Path("/content/drive/MyDrive")
AI_STUDIO = ROOT_DRIVE / "AI_Studio"
SETTINGS_PATH = AI_STUDIO / "settings.json"

COMFY_DIR = Path("/content/ComfyUI")
A1111_DIR = Path("/content/A1111")

MODELS_DIR = AI_STUDIO / "models"
BACKUPS_DIR = AI_STUDIO / "backups"
LOGS_DIR = AI_STUDIO / "logs"
TMP_DIR = AI_STUDIO / "tmp"

COMFY_MODELS_DIR = MODELS_DIR / "ComfyUI"
A1111_MODELS_DIR = MODELS_DIR / "A1111"
SHARED_MODELS_DIR = MODELS_DIR / "shared"

CHECKPOINTS_DIR = SHARED_MODELS_DIR / "checkpoints"
CONTROLNET_DIR = SHARED_MODELS_DIR / "controlnet"
ANIMATEDIFF_DIR = SHARED_MODELS_DIR / "animatediff"
INSIGHTFACE_DIR = SHARED_MODELS_DIR / "insightface"
SVD_DIR = SHARED_MODELS_DIR / "svd"
LORA_DIR = SHARED_MODELS_DIR / "loras"
VAE_DIR = SHARED_MODELS_DIR / "vae"
EMBEDDINGS_DIR = SHARED_MODELS_DIR / "embeddings"
UPSCALE_DIR = SHARED_MODELS_DIR / "upscale_models"

DEFAULT_SETTINGS = {
    "last_mode": "safe",
    "default_model": "sd15.safetensors",
    "preferred_resolution": "512x512",
    "preferred_sampler": "euler",
    "preferred_scheduler": "normal",
    "hf_token": "",
    "ports": {
        "comfyui": 8188,
        "a1111": 7860,
    },
    "repo_urls": {
        "comfyui": "https://github.com/Comfy-Org/ComfyUI.git",
        "a1111": "https://github.com/AUTOMATIC1111/stable-diffusion-webui.git",
    }
}

REPO_DEFAULTS = {
    "manager": "https://github.com/Comfy-Org/ComfyUI-Manager.git",
    "was_suite": "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    "impact_pack": "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "animatediff": "https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git",
    "reactor_comfy": "https://github.com/Gourieff/ComfyUI-ReActor.git",
    "controlnet_aux": "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "a1111_reactor": "https://github.com/Gourieff/sd-webui-reactor-sfw.git",
    "a1111_controlnet": "https://github.com/Mikubill/sd-webui-controlnet.git",
}

for p in [
    AI_STUDIO, BACKUPS_DIR, LOGS_DIR, TMP_DIR,
    MODELS_DIR, COMFY_MODELS_DIR, A1111_MODELS_DIR, SHARED_MODELS_DIR,
    CHECKPOINTS_DIR, CONTROLNET_DIR, ANIMATEDIFF_DIR, INSIGHTFACE_DIR,
    SVD_DIR, LORA_DIR, VAE_DIR, EMBEDDINGS_DIR, UPSCALE_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

def load_settings():
    if SETTINGS_PATH.exists():
        try:
            with open(SETTINGS_PATH, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            raise RuntimeError(f"Failed to load settings.json: {e}")
        merged = json.loads(json.dumps(DEFAULT_SETTINGS))
        def deep_merge(dst, src):
            for k, v in src.items():
                if isinstance(v, dict) and isinstance(dst.get(k), dict):
                    deep_merge(dst[k], v)
                else:
                    dst[k] = v
        deep_merge(merged, data)
        return merged
    else:
        save_settings(DEFAULT_SETTINGS)
        return json.loads(json.dumps(DEFAULT_SETTINGS))

def save_settings(settings: dict):
    SETTINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(SETTINGS_PATH, "w", encoding="utf-8") as f:
        json.dump(settings, f, indent=2)

SETTINGS = load_settings()

def refresh_settings():
    global SETTINGS
    SETTINGS = load_settings()
    return SETTINGS

def now_stamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def run_cmd(cmd, cwd=None, env=None, check=True):
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=cwd, env=env, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result

def run_cmd_capture(cmd, cwd=None, env=None, check=True):
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result

def safe_unlink_or_remove(path: Path):
    path = Path(path)
    if not path.exists() and not path.is_symlink():
        return
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path)

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def human_size(num_bytes: int):
    units = ["B", "KB", "MB", "GB", "TB"]
    v = float(num_bytes)
    for unit in units:
        if v < 1024 or unit == units[-1]:
            return f"{v:.2f} {unit}"
        v /= 1024

def is_port_open(port: int, host="127.0.0.1", timeout=1.0):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0

def wait_for_port(port: int, host="127.0.0.1", timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        if is_port_open(port, host=host):
            return True
        time.sleep(1)
    return False

def print_clickable(label: str, url: str):
    print(f"{label}: {url}")
    if display and HTML:
        display(HTML(f'<a href="{url}" target="_blank">{label}</a>'))

def ensure_git():
    run_cmd(["apt-get", "update", "-qq"])
    run_cmd(["apt-get", "install", "-y", "-qq", "git", "git-lfs", "aria2", "rsync"])

def pip_install(*packages):
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *packages])

def ensure_basic_python_deps():
    pip_install("requests", "packaging", "huggingface_hub", "safetensors")

def git_clone_fresh(repo_url: str, dest: Path, branch: str | None = None):
    dest = Path(dest)
    if dest.exists() or dest.is_symlink():
        print(f"Removing existing path: {dest}")
        safe_unlink_or_remove(dest)
    cmd = ["git", "clone", "--depth", "1"]
    if branch:
        cmd += ["--branch", branch]
    cmd += [repo_url, str(dest)]
    run_cmd(cmd)

def git_pull_if_repo(dest: Path):
    git_dir = Path(dest) / ".git"
    if not git_dir.exists():
        raise RuntimeError(f"Not a git repo: {dest}")
    run_cmd(["git", "pull"], cwd=str(dest))

def create_or_replace_symlink(target: Path, link_path: Path):
    target = Path(target)
    link_path = Path(link_path)
    if not target.exists():
        raise RuntimeError(f"Symlink target does not exist: {target}")
    if link_path.is_symlink() or link_path.exists():
        safe_unlink_or_remove(link_path)
    link_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(target), str(link_path))
    if not link_path.is_symlink():
        raise RuntimeError(f"Failed to create symlink: {link_path}")
    resolved = link_path.resolve()
    if resolved != target.resolve():
        raise RuntimeError(f"Broken or incorrect symlink: {link_path} -> {resolved} (expected {target.resolve()})")
    print(f"Linked: {link_path} -> {target}")

def hf_download(repo_id: str, filename: str, local_dir: Path, token: str = ""):
    from huggingface_hub import hf_hub_download
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading from Hugging Face: {repo_id}/{filename}")
    return hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=str(local_dir),
        local_dir_use_symlinks=False,
        token=token or None,
        resume_download=True,
    )

def direct_download(url: str, dest: Path, token: str = ""):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")

    headers = {}
    if token and "huggingface.co" in url:
        headers["Authorization"] = f"Bearer {token}"

    print(f"Downloading: {url}")
    with requests.get(url, stream=True, headers=headers, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        pct = (downloaded / total) * 100
                        print(f"\r  {human_size(downloaded)} / {human_size(total)} ({pct:.1f}%)", end="")
                    else:
                        print(f"\r  {human_size(downloaded)}", end="")
    print()
    tmp.replace(dest)
    return str(dest)

def download_with_manifest_item(item: dict, force: bool = False):
    path = Path(item["dest"])
    token = SETTINGS.get("hf_token", "") or ""
    if path.exists() and not force:
        print(f"Already exists, skipping: {path.name}")
        return path

    path.parent.mkdir(parents=True, exist_ok=True)

    if "repo_id" in item and "filename" in item:
        try:
            downloaded = hf_download(item["repo_id"], item["filename"], path.parent, token=token)
            downloaded_path = Path(downloaded)
            if downloaded_path.name != path.name:
                if path.exists():
                    path.unlink()
                shutil.move(str(downloaded_path), str(path))
            return path
        except Exception as e:
            if "url" not in item:
                raise RuntimeError(f"HF download failed for {item['name']}: {e}")
            print(f"HF download failed for {item['name']}, falling back to direct URL.\nDetails: {e}")

    if "url" in item:
        direct_download(item["url"], path, token=token)
        return path

    raise RuntimeError(f"No valid download method found for: {item['name']}")

RUNTIME = {
    "processes": {},
    "watchdogs": {},
    "urls": {},
}

print("Paths initialized.")
print(f"AI_STUDIO: {AI_STUDIO}")
print(f"SETTINGS_PATH: {SETTINGS_PATH}")
print(f"COMFY_DIR: {COMFY_DIR}")
print(f"A1111_DIR: {A1111_DIR}")
print(f"Default model: {SETTINGS['default_model']}")

Paths initialized.
AI_STUDIO: /content/drive/MyDrive/AI_Studio
SETTINGS_PATH: /content/drive/MyDrive/AI_Studio/settings.json
COMFY_DIR: /content/ComfyUI
A1111_DIR: /content/A1111
Default model: sd15.safetensors


## Repository Sync

Clone or update the `ai-studio-colab` repository from **GitHub** into the Colab runtime before bootstrap validation.

- **Remote:** `https://github.com/morecolorfulthanHD/ai-studio-colab.git`
- **Target:** `/content/ai-studio-colab`

Google Drive is not the notebook source of truth. This cell ensures the runtime uses the latest GitHub code.

In [ ]:
# ============================================================
# Repository Sync — Clone or Update ai-studio-colab
# ============================================================

import shutil
import subprocess
from pathlib import Path

REPO_REMOTE = "https://github.com/morecolorfulthanHD/ai-studio-colab.git"
REPO_PATH = Path("/content/ai-studio-colab")


def _run_git(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    return result


def _print_repo_state(path: Path) -> None:
    commit = _run_git(["git", "-C", str(path), "rev-parse", "--short", "HEAD"])
    branch = _run_git(["git", "-C", str(path), "rev-parse", "--abbrev-ref", "HEAD"])
    if commit.returncode != 0:
        raise RuntimeError(f"Failed to read current commit.\n{commit.stderr}")
    if branch.returncode != 0:
        raise RuntimeError(f"Failed to read current branch.\n{branch.stderr}")
    print(f"Current commit: {commit.stdout.strip()}")
    print(f"Current branch: {branch.stdout.strip()}")


def sync_repository() -> Path:
    if shutil.which("git") is None:
        raise RuntimeError(
            "ERROR: git is not available in this runtime. "
            "Install git before repository sync."
        )

    if not REPO_PATH.exists():
        print(f"Cloning repository into {REPO_PATH} ...")
        REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
        clone = _run_git(["git", "clone", REPO_REMOTE, str(REPO_PATH)])
        if clone.returncode != 0:
            raise RuntimeError(
                "ERROR: Repository clone failed.\n"
                f"Remote: {REPO_REMOTE}\n"
                f"Details: {clone.stderr or clone.stdout}"
            )
        print("Repository cloned.")
    else:
        git_dir = REPO_PATH / ".git"
        if not git_dir.is_dir():
            raise RuntimeError(
                f"ERROR: {REPO_PATH} exists but is not a git repository.\n"
                "Remove the directory or clone to a clean path before retrying."
            )
        print(f"Repository found at {REPO_PATH}. Updating ...")
        pull = _run_git(["git", "pull"], cwd=REPO_PATH)
        if pull.returncode != 0:
            raise RuntimeError(
                "ERROR: Repository update failed (git pull).\n"
                f"Details: {pull.stderr or pull.stdout}"
            )
        print("Repository updated.")

    _print_repo_state(REPO_PATH)
    return REPO_PATH.resolve()


REPO_SYNC_PATH = sync_repository()
print(f"Repository ready: {REPO_SYNC_PATH}")

## Repository Bootstrap & Validation

Runs repo-side bootstrap scripts from `core/scripts/`. Execute **after** Repository Sync, Drive mount, and path setup (Cells 2–3).

Uses the cloned repo at `/content/ai-studio-colab` (synced from GitHub).

In [ ]:
# ============================================================
# Cell 3b — Repository Bootstrap & Validation
# ============================================================

import subprocess
import sys
from pathlib import Path

_REPO_CANDIDATES = [
    Path("/content/ai-studio-colab"),
    Path("/content/drive/MyDrive/AI_Studio/ai-studio-colab"),
    Path.cwd(),
]

def _find_repo_root() -> Path:
    for candidate in _REPO_CANDIDATES:
        script = candidate / "core" / "scripts" / "bootstrap_repo.py"
        if script.is_file():
            return candidate.resolve()
    raise SystemExit(
        "ERROR: ai-studio-colab repository not found.\n"
        "Clone the repo to /content/ai-studio-colab before running bootstrap validation."
    )

REPO_ROOT = _find_repo_root()
print(f"REPO_ROOT: {REPO_ROOT}")

_SCRIPT_NAMES = [
    "bootstrap_repo.py",
    "validate_environment.py",
    "validate_paths.py",
    "validate_manifests.py",
    "list_workflows.py",
]

_results = []
for script_name in _SCRIPT_NAMES:
    script_path = REPO_ROOT / "core" / "scripts" / script_name
    print("\n" + "=" * 60)
    print(f"Running: {script_name}")
    print("=" * 60)

    if not script_path.is_file():
        print(f"FAIL — script not found: {script_path}")
        _results.append((script_name, False))
        continue

    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=str(REPO_ROOT),
        text=True,
    )
    ok = result.returncode == 0
    print(f"\n{'PASS' if ok else 'FAIL'} — {script_name} (exit {result.returncode})")
    _results.append((script_name, ok))

print("\n" + "=" * 60)
print("Repository Bootstrap & Validation Summary")
print("=" * 60)
for name, ok in _results:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

if all(ok for _, ok in _results):
    print("\nOverall: PASS — repository bootstrap validation complete.")
else:
    print("\nOverall: FAIL — one or more checks failed (see output above).")
    print("Note: Some checks may fail before ComfyUI is installed; review messages above.")

## Runtime Platform Health

Runs the registry-driven runtime report (`core/scripts/runtime_report.py`). Execute **after** Cell 3b when `REPO_ROOT` is defined.

Answers: *"Is my AI Studio healthy?"*

In [ ]:
# ============================================================
# Cell 3c — Runtime Platform Health
# ============================================================

import subprocess
import sys
from pathlib import Path

if "REPO_ROOT" not in globals():
    _candidates = [
        Path("/content/ai-studio-colab"),
        Path("/content/drive/MyDrive/AI_Studio/ai-studio-colab"),
        Path.cwd(),
    ]
    REPO_ROOT = next(
        (p.resolve() for p in _candidates if (p / "core/scripts/runtime_report.py").is_file()),
        None,
    )
    if REPO_ROOT is None:
        raise SystemExit("ERROR: REPO_ROOT not set. Run Cell 3b first or clone the repository.")

_runtime_script = REPO_ROOT / "core/scripts/runtime_report.py"
_assets_script = REPO_ROOT / "core/scripts/validate_assets.py"
_capabilities_script = REPO_ROOT / "core/scripts/validate_capabilities.py"
_nodes_script = REPO_ROOT / "core/scripts/check_nodes.py"
_models_script = REPO_ROOT / "core/scripts/verify_models.py"
_install_nodes_script = REPO_ROOT / "core/comfyui/install_nodes.py"
_install_models_script = REPO_ROOT / "core/comfyui/install_models.py"
_workflow_path = REPO_ROOT / "workflows/base/txt2img/workflow.json"
print(f"Runtime report: {_runtime_script}\n")

_result = subprocess.run(
    [sys.executable, str(_runtime_script)],
    cwd=str(REPO_ROOT),
    text=True,
)

print()
_summary = subprocess.run(
    [sys.executable, str(_runtime_script), "--summary"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
if _summary.stdout.strip():
    print("Summary:", _summary.stdout.strip())

print("\n" + "=" * 60)
print("Asset validation")
print("=" * 60)
subprocess.run(
    [sys.executable, str(_assets_script), "--summary"],
    cwd=str(REPO_ROOT),
    text=True,
)

print("\n" + "=" * 60)
print("Capability summary")
print("=" * 60)
subprocess.run(
    [sys.executable, str(_capabilities_script), "--summary"],
    cwd=str(REPO_ROOT),
    text=True,
)

print("\n" + "=" * 60)
print("Node validation")
print("=" * 60)
subprocess.run(
    [sys.executable, str(_nodes_script)],
    cwd=str(REPO_ROOT),
    text=True,
)

print("\n" + "=" * 60)
print("Model validation")
print("=" * 60)
subprocess.run(
    [sys.executable, str(_models_script)],
    cwd=str(REPO_ROOT),
    text=True,
)

print("\n" + "=" * 60)
print("ComfyUI install dry-runs")
print("=" * 60)
subprocess.run(
    [sys.executable, str(_install_nodes_script), "--dry-run"],
    cwd=str(REPO_ROOT),
    text=True,
)
subprocess.run(
    [sys.executable, str(_install_models_script), "--dry-run"],
    cwd=str(REPO_ROOT),
    text=True,
)

print("\nBase txt2img workflow:", _workflow_path)
print("Exists:", _workflow_path.is_file())

if _result.returncode == 0:
    print("\nOverall: PASS — runtime platform health check complete.")
else:
    print("\nOverall: WARN/FAIL — review runtime report above.")

In [4]:
# ============================================================
# Cell 4 — Starter Models
# ============================================================

STARTER_MODELS = [
    {
        "name": "sd15",
        "dest": str(CHECKPOINTS_DIR / "sd15.safetensors"),
        "repo_id": "stable-diffusion-v1-5/stable-diffusion-v1-5",
        "filename": "v1-5-pruned-emaonly.safetensors",
        "url": "https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors",
        "min_bytes": 3_500_000_000,
    },
    {
        "name": "sdxl_base",
        "dest": str(CHECKPOINTS_DIR / "sd_xl_base_1.0.safetensors"),
        "repo_id": "stabilityai/stable-diffusion-xl-base-1.0",
        "filename": "sd_xl_base_1.0.safetensors",
        "url": "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
        "min_bytes": 6_500_000_000,
    },
    {
        "name": "sdxl_refiner",
        "dest": str(CHECKPOINTS_DIR / "sd_xl_refiner_1.0.safetensors"),
        "repo_id": "stabilityai/stable-diffusion-xl-refiner-1.0",
        "filename": "sd_xl_refiner_1.0.safetensors",
        "url": "https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/resolve/main/sd_xl_refiner_1.0.safetensors",
        "min_bytes": 5_500_000_000,
    },
    {
        "name": "svd_xt",
        "dest": str(SVD_DIR / "svd_xt.safetensors"),
        "repo_id": "stabilityai/stable-video-diffusion-img2vid-xt",
        "filename": "svd_xt.safetensors",
        "url": "https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt/resolve/main/svd_xt.safetensors",
        "min_bytes": 9_000_000_000,
    },
    {
        "name": "animatediff_mm_v2",
        "dest": str(ANIMATEDIFF_DIR / "mm_sd_v15_v2.ckpt"),
        "repo_id": "guoyww/animatediff",
        "filename": "mm_sd_v15_v2.ckpt",
        "url": "https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt",
        "min_bytes": 1_700_000_000,
    },
    {
        "name": "insightface_w600k_r50",
        "dest": str(INSIGHTFACE_DIR / "models" / "buffalo_l" / "w600k_r50.onnx"),
        "url": "https://huggingface.co/yolkailtd/face-swap-models/resolve/main/insightface/models/buffalo_l/w600k_r50.onnx",
        "min_bytes": 150_000_000,
    },
    {
        "name": "controlnet_canny",
        "dest": str(CONTROLNET_DIR / "control_v11p_sd15_canny_fp16.safetensors"),
        "repo_id": "lllyasviel/control_v11p_sd15_canny",
        "filename": "diffusion_pytorch_model.fp16.safetensors",
        "url": "https://huggingface.co/lllyasviel/control_v11p_sd15_canny/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        "min_bytes": 700_000_000,
    },
    {
        "name": "controlnet_depth",
        "dest": str(CONTROLNET_DIR / "control_v11f1p_sd15_depth_fp16.safetensors"),
        "repo_id": "lllyasviel/control_v11f1p_sd15_depth",
        "filename": "diffusion_pytorch_model.fp16.safetensors",
        "url": "https://huggingface.co/lllyasviel/control_v11f1p_sd15_depth/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        "min_bytes": 700_000_000,
    },
    {
        "name": "controlnet_lineart",
        "dest": str(CONTROLNET_DIR / "control_v11p_sd15_lineart_fp16.safetensors"),
        "repo_id": "lllyasviel/control_v11p_sd15_lineart",
        "filename": "diffusion_pytorch_model.fp16.safetensors",
        "url": "https://huggingface.co/lllyasviel/control_v11p_sd15_lineart/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        "min_bytes": 700_000_000,
    },
]

def starter_model_status():
    rows = []
    for item in STARTER_MODELS:
        p = Path(item["dest"])
        rows.append({
            "name": item["name"],
            "exists": p.exists(),
            "path": str(p),
            "size": human_size(p.stat().st_size) if p.exists() else "missing",
            "valid_size": (p.exists() and p.stat().st_size >= item["min_bytes"]),
        })
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except Exception:
        return rows

def download_starter_models(names=None, force=False):
    selected = []
    if names is None:
        selected = STARTER_MODELS
    else:
        names_set = set(names)
        selected = [x for x in STARTER_MODELS if x["name"] in names_set]

    if not selected:
        raise ValueError("No starter models selected.")

    for item in selected:
        print(f"\n=== {item['name']} ===")
        path = download_with_manifest_item(item, force=force)
        size = path.stat().st_size
        if size < item["min_bytes"]:
            raise RuntimeError(
                f"Downloaded file is smaller than expected for {item['name']}.\n"
                f"Path: {path}\nActual: {human_size(size)}\nExpected minimum: {human_size(item['min_bytes'])}"
            )
        print(f"OK: {path} ({human_size(size)})")

print("Starter model manifest loaded.")
status = starter_model_status()
status

Starter model manifest loaded.


,name,exists,path,size,valid_size
0,sd15,True,/content/drive/MyDrive/AI_Studio/models/shared...,3.97 GB,True
1,sdxl_base,True,/content/drive/MyDrive/AI_Studio/models/shared...,6.46 GB,True
2,sdxl_refiner,True,/content/drive/MyDrive/AI_Studio/models/shared...,5.66 GB,True
3,svd_xt,True,/content/drive/MyDrive/AI_Studio/models/shared...,8.90 GB,True
4,animatediff_mm_v2,True,/content/drive/MyDrive/AI_Studio/models/shared...,1.69 GB,True
5,insightface_w600k_r50,True,/content/drive/MyDrive/AI_Studio/models/shared...,166.31 MB,True
6,controlnet_canny,True,/content/drive/MyDrive/AI_Studio/models/shared...,689.12 MB,True
7,controlnet_depth,True,/content/drive/MyDrive/AI_Studio/models/shared...,689.12 MB,True
8,controlnet_lineart,True,/content/drive/MyDrive/AI_Studio/models/shared...,689.12 MB,True


In [5]:
# ============================================================
# Cell 5 — Model Manager
# ============================================================

MODEL_ROOTS = [
    CHECKPOINTS_DIR,
    CONTROLNET_DIR,
    ANIMATEDIFF_DIR,
    INSIGHTFACE_DIR,
    SVD_DIR,
    LORA_DIR,
    VAE_DIR,
    EMBEDDINGS_DIR,
    UPSCALE_DIR,
]

def list_models():
    rows = []
    for root in MODEL_ROOTS:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if path.is_file():
                rows.append({
                    "name": path.name,
                    "relative_path": str(path.relative_to(AI_STUDIO)),
                    "size": human_size(path.stat().st_size),
                    "sha256": sha256_file(path)[:16],
                })
    rows = sorted(rows, key=lambda x: x["relative_path"])
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        return df if not df.empty else "No models found."
    except Exception:
        return rows if rows else "No models found."

def show_model_hashes():
    rows = []
    for root in MODEL_ROOTS:
        for path in root.rglob("*"):
            if path.is_file():
                rows.append({
                    "path": str(path),
                    "sha256": sha256_file(path),
                })
    try:
        import pandas as pd
        return pd.DataFrame(rows) if rows else "No model files found."
    except Exception:
        return rows if rows else "No model files found."

def validate_model_sizes():
    results = []
    manifest_map = {item["dest"]: item for item in STARTER_MODELS}
    for root in MODEL_ROOTS:
        for path in root.rglob("*"):
            if path.is_file():
                info = {
                    "path": str(path),
                    "size": human_size(path.stat().st_size),
                    "valid": True,
                    "reason": "",
                }
                item = manifest_map.get(str(path))
                if item and path.stat().st_size < item["min_bytes"]:
                    info["valid"] = False
                    info["reason"] = f"Too small; expected >= {human_size(item['min_bytes'])}"
                results.append(info)

    try:
        import pandas as pd
        return pd.DataFrame(results) if results else "No model files found."
    except Exception:
        return results if results else "No model files found."

def set_default_model(model_filename: str):
    candidates = list(CHECKPOINTS_DIR.rglob(model_filename))
    if not candidates:
        raise FileNotFoundError(f"Model not found in checkpoints: {model_filename}")
    SETTINGS["default_model"] = model_filename
    save_settings(SETTINGS)
    print(f"Default model set to: {model_filename}")

def delete_model(rel_or_abs_path: str):
    path = Path(rel_or_abs_path)
    if not path.is_absolute():
        path = AI_STUDIO / rel_or_abs_path
    if not path.exists():
        raise FileNotFoundError(f"Model not found: {path}")
    if not path.is_file():
        raise RuntimeError(f"Refusing to delete non-file path: {path}")
    path.unlink()
    print(f"Deleted: {path}")

def download_model_custom(name: str, url: str, subdir: str = "shared/checkpoints"):
    dest_dir = MODELS_DIR / subdir
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / name
    direct_download(url, dest, token=SETTINGS.get("hf_token", ""))
    print(f"Downloaded custom model to: {dest}")

def model_manager_menu():
    while True:
        print("\n=== Model Manager ===")
        print("1. List models")
        print("2. Download starter model(s)")
        print("3. Download custom model from URL")
        print("4. Delete model")
        print("5. Validate model sizes")
        print("6. Show model hashes")
        print("7. Set default model")
        print("0. Back")
        choice = input("Select: ").strip()

        try:
            if choice == "1":
                print(list_models())
            elif choice == "2":
                print("Available starter models:", ", ".join([x["name"] for x in STARTER_MODELS]))
                raw = input("Enter comma-separated names or ALL: ").strip()
                if raw.upper() == "ALL":
                    download_starter_models()
                else:
                    names = [x.strip() for x in raw.split(",") if x.strip()]
                    download_starter_models(names=names)
            elif choice == "3":
                name = input("Filename to save as: ").strip()
                url = input("Direct URL: ").strip()
                subdir = input("Subdir under models (default shared/checkpoints): ").strip() or "shared/checkpoints"
                download_model_custom(name, url, subdir=subdir)
            elif choice == "4":
                target = input("Relative or absolute file path: ").strip()
                delete_model(target)
            elif choice == "5":
                print(validate_model_sizes())
            elif choice == "6":
                print(show_model_hashes())
            elif choice == "7":
                model_filename = input("Checkpoint filename (e.g. sd15.safetensors): ").strip()
                set_default_model(model_filename)
            elif choice == "0":
                break
            else:
                print("Invalid selection.")
        except Exception as e:
            print(f"ERROR: {e}")

print("Model Manager ready.")

Model Manager ready.


In [6]:
# ============================================================
# Cell 6 — Node Manager
# ============================================================

DEFAULT_COMFY_NODES = {
    "ComfyUI-Manager": REPO_DEFAULTS["manager"],
    "ComfyUI-Impact-Pack": REPO_DEFAULTS["impact_pack"],
    "ComfyUI-AnimateDiff-Evolved": REPO_DEFAULTS["animatediff"],
    "ComfyUI-ReActor": REPO_DEFAULTS["reactor_comfy"],
    "comfyui_controlnet_aux": REPO_DEFAULTS["controlnet_aux"],
    "was-node-suite-comfyui": REPO_DEFAULTS["was_suite"],
}

def comfy_custom_nodes_dir():
    return COMFY_DIR / "custom_nodes"

def list_nodes():
    root = comfy_custom_nodes_dir()
    if not root.exists():
        return "ComfyUI custom_nodes folder not found yet. Install ComfyUI first."
    rows = []
    for item in sorted(root.iterdir()):
        if item.is_dir():
            rows.append({
                "name": item.name,
                "git_repo": (item / ".git").exists(),
                "has_requirements": (item / "requirements.txt").exists(),
                "has_install_py": (item / "install.py").exists(),
            })
    try:
        import pandas as pd
        return pd.DataFrame(rows) if rows else "No custom nodes found."
    except Exception:
        return rows if rows else "No custom nodes found."

def install_requirements_for_node(node_path: Path):
    node_path = Path(node_path)
    if (node_path / "requirements.txt").exists():
        run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(node_path / "requirements.txt")])
    if (node_path / "install.py").exists():
        run_cmd([sys.executable, str(node_path / "install.py")], cwd=str(node_path), check=False)

def install_node(repo_url: str, name: str | None = None):
    if not COMFY_DIR.exists():
        raise RuntimeError("Install ComfyUI before installing nodes.")
    root = comfy_custom_nodes_dir()
    root.mkdir(parents=True, exist_ok=True)

    if name is None:
        name = repo_url.rstrip("/").split("/")[-1]
        if name.endswith(".git"):
            name = name[:-4]

    dest = root / name
    git_clone_fresh(repo_url, dest)
    install_requirements_for_node(dest)
    print(f"Installed node: {name}")

def remove_node(name: str):
    dest = comfy_custom_nodes_dir() / name
    if not dest.exists():
        raise FileNotFoundError(f"Node not found: {dest}")
    safe_unlink_or_remove(dest)
    print(f"Removed node: {name}")

def validate_nodes():
    root = comfy_custom_nodes_dir()
    if not root.exists():
        return "ComfyUI custom_nodes folder not found."
    rows = []
    for item in sorted(root.iterdir()):
        if item.is_dir():
            ok = True
            reason = ""
            if not (item / ".git").exists():
                ok = False
                reason = "Missing .git folder"
            rows.append({
                "node": item.name,
                "valid": ok,
                "reason": reason,
            })
    try:
        import pandas as pd
        return pd.DataFrame(rows) if rows else "No nodes installed."
    except Exception:
        return rows if rows else "No nodes installed."

def reset_nodes(install_defaults=True):
    root = comfy_custom_nodes_dir()
    if root.exists():
        for item in root.iterdir():
            if item.name == "__pycache__":
                continue
            safe_unlink_or_remove(item)
    root.mkdir(parents=True, exist_ok=True)

    if install_defaults:
        # Install WAS last because it can change package versions.
        ordered = [
            "ComfyUI-Manager",
            "ComfyUI-Impact-Pack",
            "ComfyUI-AnimateDiff-Evolved",
            "ComfyUI-ReActor",
            "comfyui_controlnet_aux",
            "was-node-suite-comfyui",
        ]
        for node_name in ordered:
            install_node(DEFAULT_COMFY_NODES[node_name], name=node_name)
    print("Node reset complete.")

def node_manager_menu():
    while True:
        print("\n=== Node Manager ===")
        print("1. List nodes")
        print("2. Install default stable nodes")
        print("3. Install custom node from URL")
        print("4. Remove node")
        print("5. Validate nodes")
        print("6. Reset nodes")
        print("0. Back")
        choice = input("Select: ").strip()

        try:
            if choice == "1":
                print(list_nodes())
            elif choice == "2":
                for node_name, repo_url in DEFAULT_COMFY_NODES.items():
                    if node_name == "was-node-suite-comfyui":
                        continue
                    install_node(repo_url, name=node_name)
                install_node(DEFAULT_COMFY_NODES["was-node-suite-comfyui"], name="was-node-suite-comfyui")
            elif choice == "3":
                repo_url = input("Git repo URL: ").strip()
                custom_name = input("Folder name (optional): ").strip() or None
                install_node(repo_url, name=custom_name)
            elif choice == "4":
                name = input("Installed node folder name: ").strip()
                remove_node(name)
            elif choice == "5":
                print(validate_nodes())
            elif choice == "6":
                raw = input("Reinstall default nodes after reset? (y/n): ").strip().lower()
                reset_nodes(install_defaults=(raw == "y"))
            elif choice == "0":
                break
            else:
                print("Invalid selection.")
        except Exception as e:
            print(f"ERROR: {e}")

print("Node Manager ready.")

Node Manager ready.


In [7]:
# ============================================================
# Cell 7 — Backup & Restore
# ============================================================

def backup_ai_studio(scope="all"):
    ts = now_stamp()
    if scope == "all":
        src = AI_STUDIO
        archive = BACKUPS_DIR / f"AI_Studio_backup_{ts}.tar.gz"
    elif scope == "models":
        src = MODELS_DIR
        archive = BACKUPS_DIR / f"AI_Studio_models_{ts}.tar.gz"
    elif scope == "nodes":
        src = AI_STUDIO / "snapshots" / "nodes"
        archive = BACKUPS_DIR / f"AI_Studio_nodes_{ts}.tar.gz"
        src.mkdir(parents=True, exist_ok=True)
    else:
        raise ValueError("scope must be one of: all, models, nodes")

    archive.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "w:gz") as tar:
        tar.add(src, arcname=src.name)
    print(f"Backup created: {archive}")
    return archive

def snapshot_current_nodes():
    snap_dir = AI_STUDIO / "snapshots" / "nodes"
    safe_unlink_or_remove(snap_dir)
    snap_dir.mkdir(parents=True, exist_ok=True)
    src = comfy_custom_nodes_dir()
    if src.exists():
        shutil.copytree(src, snap_dir / "custom_nodes", dirs_exist_ok=True)
    print(f"Node snapshot saved to: {snap_dir}")

def list_backups():
    archives = sorted(BACKUPS_DIR.glob("*.tar.gz"))
    rows = []
    for a in archives:
        rows.append({
            "file": a.name,
            "size": human_size(a.stat().st_size),
            "modified": datetime.fromtimestamp(a.stat().st_mtime).isoformat(timespec="seconds"),
        })
    try:
        import pandas as pd
        return pd.DataFrame(rows) if rows else "No backups found."
    except Exception:
        return rows if rows else "No backups found."

def restore_backup(archive_name_or_path: str, target_dir: str | None = None):
    archive = Path(archive_name_or_path)
    if not archive.is_absolute():
        archive = BACKUPS_DIR / archive_name_or_path
    if not archive.exists():
        raise FileNotFoundError(f"Backup not found: {archive}")

    destination = Path(target_dir) if target_dir else AI_STUDIO.parent
    destination.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(path=destination)
    print(f"Restored {archive} into {destination}")

def backup_restore_menu():
    while True:
        print("\n=== Backup & Restore ===")
        print("1. Backup entire AI_Studio")
        print("2. Backup only models")
        print("3. Snapshot + backup nodes")
        print("4. List backups")
        print("5. Restore selected backup")
        print("0. Back")
        choice = input("Select: ").strip()

        try:
            if choice == "1":
                backup_ai_studio("all")
            elif choice == "2":
                backup_ai_studio("models")
            elif choice == "3":
                snapshot_current_nodes()
                backup_ai_studio("nodes")
            elif choice == "4":
                print(list_backups())
            elif choice == "5":
                print(list_backups())
                name = input("Backup filename: ").strip()
                restore_backup(name)
            elif choice == "0":
                break
            else:
                print("Invalid selection.")
        except Exception as e:
            print(f"ERROR: {e}")

print("Backup & Restore tools ready.")

Backup & Restore tools ready.


In [8]:
# ============================================================
# Cell 8 — Full Reset
# ============================================================

def stop_process(name: str):
    proc = RUNTIME["processes"].get(name)
    if proc and proc.poll() is None:
        print(f"Stopping {name}...")
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
    RUNTIME["processes"].pop(name, None)
    RUNTIME["watchdogs"].pop(name, None)
    RUNTIME["urls"].pop(name, None)

def full_reset_runtime_only():
    stop_process("comfyui")
    stop_process("a1111")

    for path in [COMFY_DIR, A1111_DIR]:
        if path.exists() or path.is_symlink():
            print(f"Removing runtime folder: {path}")
            safe_unlink_or_remove(path)

    # Clean any obvious runtime symlinks left behind.
    candidate_links = [
        Path("/content/ComfyUI"),
        Path("/content/A1111"),
    ]
    for p in candidate_links:
        if p.is_symlink():
            print(f"Removing leftover symlink: {p}")
            p.unlink()

    print("Full reset complete.")
    print("Drive-backed models were not deleted.")

# Uncomment to run immediately:
# full_reset_runtime_only()
print("Reset helper ready. Run full_reset_runtime_only() when needed.")

Reset helper ready. Run full_reset_runtime_only() when needed.


In [9]:
# ============================================================
# Cell 9 — Installers
# ============================================================

def install_comfyui(mode="safe"):
    mode = mode.lower()
    ensure_git()
    ensure_basic_python_deps()

    print(f"Installing ComfyUI in {mode} mode...")
    git_clone_fresh(SETTINGS["repo_urls"]["comfyui"], COMFY_DIR)

    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_DIR / "requirements.txt")])

    # Drive-backed model links
    create_or_replace_symlink(SHARED_MODELS_DIR, COMFY_DIR / "models")

    # Optional node installs
    if mode == "full":
        root = comfy_custom_nodes_dir()
        root.mkdir(parents=True, exist_ok=True)

        install_order = [
            ("ComfyUI-Manager", REPO_DEFAULTS["manager"]),
            ("ComfyUI-Impact-Pack", REPO_DEFAULTS["impact_pack"]),
            ("ComfyUI-AnimateDiff-Evolved", REPO_DEFAULTS["animatediff"]),
            ("ComfyUI-ReActor", REPO_DEFAULTS["reactor_comfy"]),
            ("comfyui_controlnet_aux", REPO_DEFAULTS["controlnet_aux"]),
            ("was-node-suite-comfyui", REPO_DEFAULTS["was_suite"]),
        ]
        for name, repo in install_order:
            install_node(repo, name=name)

    print("ComfyUI install complete.")
    return COMFY_DIR

def install_a1111(mode="full"):
    ensure_git()
    ensure_basic_python_deps()

    print("Installing A1111...")
    git_clone_fresh(SETTINGS["repo_urls"]["a1111"], A1111_DIR)

    # Shared model links
    (A1111_DIR / "models").mkdir(parents=True, exist_ok=True)
    create_or_replace_symlink(CHECKPOINTS_DIR, A1111_DIR / "models" / "Stable-diffusion")
    create_or_replace_symlink(CONTROLNET_DIR, A1111_DIR / "models" / "ControlNet")
    create_or_replace_symlink(VAE_DIR, A1111_DIR / "models" / "VAE")
    create_or_replace_symlink(EMBEDDINGS_DIR, A1111_DIR / "embeddings")
    create_or_replace_symlink(LORA_DIR, A1111_DIR / "models" / "Lora")

    ext_dir = A1111_DIR / "extensions"
    ext_dir.mkdir(parents=True, exist_ok=True)

    git_clone_fresh(REPO_DEFAULTS["a1111_reactor"], ext_dir / "sd-webui-reactor-sfw")
    git_clone_fresh(REPO_DEFAULTS["a1111_controlnet"], ext_dir / "sd-webui-controlnet")

    # Install extension requirements if present
    for ext in [ext_dir / "sd-webui-reactor-sfw", ext_dir / "sd-webui-controlnet"]:
        if (ext / "requirements.txt").exists():
            run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(ext / "requirements.txt")], cwd=str(ext))
        if (ext / "install.py").exists():
            run_cmd([sys.executable, str(ext / "install.py")], cwd=str(ext), check=False)

    print("A1111 install complete.")
    return A1111_DIR

def install_mode(mode="safe"):
    mode = mode.lower()
    if mode not in {"safe", "minimal", "full"}:
        raise ValueError("Mode must be safe, minimal, or full")

    SETTINGS["last_mode"] = mode
    save_settings(SETTINGS)

    if mode == "safe":
        install_comfyui(mode="safe")
    elif mode == "minimal":
        install_comfyui(mode="minimal")
        needed = ["sd15"]
        download_starter_models(names=needed)
    elif mode == "full":
        install_comfyui(mode="full")
        install_a1111(mode="full")
        download_starter_models(names=[
            "sd15",
            "sdxl_base",
            "sdxl_refiner",
            "svd_xt",
            "animatediff_mm_v2",
            "insightface_w600k_r50",
            "controlnet_canny",
            "controlnet_depth",
            "controlnet_lineart",
        ])

    print(f"Mode install finished: {mode}")

print("Installers ready.")


Installers ready.


In [10]:
# ============================================================
# Cell 10 — Launchers
# ============================================================

from google.colab.output import eval_js

SD15_CHECKPOINT = Path("/content/drive/MyDrive/AI_Studio/models/shared/checkpoints/sd15.safetensors")
COMFYUI_OUTPUT_DIR = Path("/content/ComfyUI/output")


def resolve_repo_root_for_launch() -> Path:
    if "REPO_ROOT" in globals():
        try:
            root = Path(REPO_ROOT).resolve()
            if (root / "core/scripts/runtime_report.py").is_file():
                return root
        except Exception:
            pass
    for candidate in (
        Path("/content/ai-studio-colab"),
        Path("/content/drive/MyDrive/AI_Studio/ai-studio-colab"),
        Path.cwd(),
    ):
        if (candidate / "core/scripts/runtime_report.py").is_file():
            return candidate.resolve()
    raise RuntimeError(
        "Repository not found. Run Repository Sync (before Cell 3b) before launching."
    )


def _format_script_failure(script_rel: str, result: subprocess.CompletedProcess) -> str:
    lines = [f"Script failed (exit {result.returncode}): {script_rel}"]
    if result.stdout and result.stdout.strip():
        lines.append("--- stdout ---")
        lines.append(result.stdout.rstrip())
    if result.stderr and result.stderr.strip():
        lines.append("--- stderr ---")
        lines.append(result.stderr.rstrip())
    return "\n".join(lines)


def run_repo_python(script_rel: str, *args: str, check: bool = True):
    repo_root = resolve_repo_root_for_launch()
    script = repo_root / script_rel
    if not script.is_file():
        raise RuntimeError(f"Repo script not found: {script}")
    cmd = [sys.executable, str(script), *args]
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=str(repo_root), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    if check and result.returncode != 0:
        raise RuntimeError(_format_script_failure(script_rel, result))
    return result


def run_repo_bash(script_rel: str, *args: str, check: bool = True):
    repo_root = resolve_repo_root_for_launch()
    script = repo_root / script_rel
    if not script.is_file():
        raise RuntimeError(f"Repo script not found: {script}")
    cmd = ["bash", str(script), *args]
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=str(repo_root), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    if check and result.returncode != 0:
        raise RuntimeError(_format_script_failure(script_rel, result))
    return result


def proxy_url_for_port(port: int) -> str:
    return eval_js(f"google.colab.kernel.proxyPort({port})")


def start_watchdog(name: str, launch_fn):
    if name in RUNTIME["watchdogs"]:
        print(f"Watchdog already registered for {name}")
        return

    def _runner():
        while True:
            time.sleep(5)
            proc = RUNTIME["processes"].get(name)
            if proc is None:
                break
            ret = proc.poll()
            if ret is not None:
                print(f"[watchdog] {name} exited with code {ret}. Restarting...")
                try:
                    launch_fn(restarting=True)
                except Exception as e:
                    print(f"[watchdog] failed to restart {name}: {e}")
                    time.sleep(10)

    t = threading.Thread(target=_runner, daemon=True)
    t.start()
    RUNTIME["watchdogs"][name] = t
    print(f"Watchdog started for {name}")


def initialize_runtime_identity():
    """Create this VM's identity in ephemeral /content runtime storage."""
    result = run_repo_python(
        "core/scripts/run_output_watcher.py",
        "--initialize-runtime",
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError("Unable to initialize current Colab runtime identity.")


def _read_current_watcher_status():
    status_path = Path("/content/ai-studio-runtime/output-watcher/watcher_status.json")
    if not status_path.is_file():
        return None
    try:
        return json.loads(status_path.read_text(encoding="utf-8"))
    except Exception:
        return None


def _watcher_status_healthy(payload) -> bool:
    return bool(
        isinstance(payload, dict)
        and payload.get("watcher") == "OK"
        and payload.get("ownership_state") == "current_runtime"
        and payload.get("process_alive") is True
        and payload.get("process_command_valid") is True
        and payload.get("heartbeat_fresh") is True
        and payload.get("last_history_poll")
    )


def wait_for_output_watcher_ready(proc, timeout=30):
    deadline = time.time() + timeout
    while time.time() < deadline:
        payload = _read_current_watcher_status()
        if _watcher_status_healthy(payload):
            return payload
        if proc.poll() not in (None, 0):
            break
        time.sleep(0.5)
    return None


def launch_comfyui(restarting=False):
    if not COMFY_DIR.exists():
        raise RuntimeError("ComfyUI is not installed. Run Launch (option 1) first.")

    # Runtime identity is ephemeral and is created before any watcher ownership check.
    initialize_runtime_identity()
    stop_process("comfyui")

    port = int(SETTINGS["ports"]["comfyui"])
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    cmd = [
        sys.executable,
        "main.py",
        "--listen", "127.0.0.1",
        "--port", str(port),
    ]

    log_path = LOGS_DIR / "comfyui.log"
    log_file = open(log_path, "a", encoding="utf-8")

    print("Launching ComfyUI...")
    proc = subprocess.Popen(
        cmd,
        cwd=str(COMFY_DIR),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    RUNTIME["processes"]["comfyui"] = proc

    if not wait_for_port(port, timeout=300):
        raise RuntimeError(f"ComfyUI did not open port {port} in time. Check {log_path}")

    url = proxy_url_for_port(port)
    RUNTIME["urls"]["comfyui"] = url
    print_clickable("ComfyUI URL", url)

    if not restarting:
        start_watchdog("comfyui", launch_comfyui)
    start_output_autosync_watcher()


def start_output_autosync_watcher():
    """Start or validate exactly one current-runtime watcher."""
    script = REPO_ROOT / "core" / "scripts" / "run_output_watcher.py"
    if not script.is_file():
        print(f"OutputWatcher: FAIL — watcher script missing: {script}")
        return
    log_path = LOGS_DIR / "output_watcher.log"
    log_file = open(log_path, "a", encoding="utf-8")
    # start_new_session keeps the watcher independent while the notebook sits at Select:.
    proc = subprocess.Popen(
        [sys.executable, str(script), "--repo-root", str(REPO_ROOT)],
        cwd=str(REPO_ROOT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    RUNTIME["output_watcher_proc"] = proc
    status = wait_for_output_watcher_ready(proc)
    if status is None:
        print(f"OutputWatcher: FAIL — no verified current-runtime heartbeat. Logs: {log_path}")
        return
    print(
        f"OutputWatcher: OK — pid {status.get('watcher_pid')}, "
        f"runtime {status.get('runtime_id')}, heartbeat {status.get('heartbeat')}"
    )
    print("Watcher runs independently — leaving the control panel at Select: is supported; option 0 is not required.")


def launch_a1111(restarting=False):
    if not A1111_DIR.exists():
        raise RuntimeError("A1111 is not installed. Run Cell 9 installer first.")

    stop_process("a1111")

    port = int(SETTINGS["ports"]["a1111"])
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["COMMANDLINE_ARGS"] = (
        f"--listen --port {port} --api --skip-python-version-check "
        f"--enable-insecure-extension-access --no-download-sd-model"
    )

    log_path = LOGS_DIR / "a1111.log"
    log_file = open(log_path, "a", encoding="utf-8")

    print("Launching A1111...")
    proc = subprocess.Popen(
        [sys.executable, "launch.py"],
        cwd=str(A1111_DIR),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    RUNTIME["processes"]["a1111"] = proc

    if not wait_for_port(port, timeout=600):
        raise RuntimeError(f"A1111 did not open port {port} in time. Check {log_path}")

    url = proxy_url_for_port(port)
    RUNTIME["urls"]["a1111"] = url
    print_clickable("A1111 URL", url)

    if not restarting:
        start_watchdog("a1111", launch_a1111)


def check_sd15_checkpoint(warn_only: bool = True) -> bool:
    print("\nSD1.5 checkpoint validation")
    print("=" * 40)
    run_repo_python("core/scripts/verify_models.py", check=False)
    if SD15_CHECKPOINT.is_file() and SD15_CHECKPOINT.stat().st_size > 0:
        print(f"OK: SD1.5 present at {SD15_CHECKPOINT}")
        return True

    print("MISSING: SD1.5 checkpoint not found.")
    print(f"  Expected path: {SD15_CHECKPOINT}")
    print("  Place sd15.safetensors at this path before txt2img can run.")
    print("  No automatic download is performed.")
    if warn_only:
        print("  Launch continues, but txt2img generation will not be ready.")
    return False


def print_automatic_output_persistence():
    repo = resolve_repo_root_for_launch()
    drive_outputs = "/content/drive/MyDrive/AI_Studio/outputs"
    evidence = "/content/drive/MyDrive/AI_Studio/logs/generation_evidence.jsonl"
    status = _read_current_watcher_status()
    watcher = "OK (current runtime)" if _watcher_status_healthy(status) else "FAIL (not verified)"
    print("\nAutomatic output persistence:")
    print(f"  OutputWatcher: {watcher}")
    if status:
        print(f"  Runtime ID: {status.get('runtime_id')}")
        print(f"  Heartbeat: {status.get('heartbeat')}")
    print(f"  Drive destination: {drive_outputs}")
    print(f"  Evidence ledger: {evidence}")
    print("After clicking Run in ComfyUI:")
    print("  - completed outputs are detected automatically")
    print("  - Drive copies are verified by byte size and SHA-256")
    print("  - generation evidence is updated automatically")
    print("  - no manual sync command is required")
    print(f"  (Maintenance only: {repo / 'core/scripts/sync_outputs.py'} , {repo / 'core/scripts/verify_generation.py'})")


def print_txt2img_workflow_guidance():
    workflow = resolve_repo_root_for_launch() / "workflows/base/txt2img/workflow.json"
    print("\nBase txt2img workflow:")
    print(f"  {workflow}")
    print("\nNext steps:")
    print("  1. Open the ComfyUI URL above")
    print("  2. Import or drag workflow.json into ComfyUI")
    print("  3. Confirm checkpoint = sd15.safetensors")
    print("  4. Click Queue Prompt / Run")
    print(f"  5. Outputs sync automatically to Drive (no manual sync command)")
    print_automatic_output_persistence()


def print_post_launch_runtime_summary():
    print("\nPost-launch runtime summary")
    print("=" * 40)
    run_repo_python("core/scripts/runtime_report.py", "--summary", check=False)
    run_repo_python(
        "core/scripts/validate_capabilities.py",
        "--capability",
        "txt2img",
        "--summary",
        check=False,
    )
    run_repo_python("core/scripts/report_generation_history.py", "--summary", check=False)
    print("\nNote: txt2img readiness and generation evidence are computed dynamically.")


def studio_pre_launch_validation():
    print("\nPre-launch dogfood check")
    print("=" * 40)
    run_repo_python("core/scripts/dogfood_core_runtime.py", check=False)


def studio_launch(mode: str = "safe"):
    mode = mode.lower().strip()
    if mode not in {"safe", "minimal", "full"}:
        raise ValueError("Mode must be safe, minimal, or full")

    SETTINGS["last_mode"] = mode
    save_settings(SETTINGS)

    print("\n" + "=" * 60)
    print(f"AI Studio Launch — {mode.upper()} mode")
    print("=" * 60)

    studio_pre_launch_validation()

    print("\nComfyUI install/validation")
    print("=" * 40)
    run_repo_bash("core/comfyui/install.sh", "--execute")

    if mode == "full":
        print("\nCustom node install (full mode)")
        print("=" * 40)
        run_repo_python("core/comfyui/install_nodes.py", "--execute")
        run_repo_python("core/scripts/check_nodes.py", check=False)
    else:
        print("\nSkipping custom node install (safe/minimal mode).")

    sd15_ok = check_sd15_checkpoint(warn_only=True)
    if mode in {"minimal", "full"} and not sd15_ok:
        print("\nWARN: SD1.5 is required for base txt2img generation readiness.")

    launch_comfyui()

    if mode in {"minimal", "full"}:
        print_txt2img_workflow_guidance()

    print_post_launch_runtime_summary()

    print("\nPost-launch dogfood check")
    print("=" * 40)
    run_repo_python("core/scripts/dogfood_core_runtime.py", check=False)

    print("\nLaunch complete.")


DRIVE_INPUTS_DIR = Path("/content/drive/MyDrive/AI_Studio/inputs")
RUNTIME_WORKFLOWS_DIR = Path("/content/ai-studio-runtime/workflows")


def image_editing_menu():
    while True:
        print("\n=== Image Editing ===")
        print("1. img2img")
        print("2. Inpainting")
        print("3. Outpainting")
        print("0. Back")
        choice = input("Select: ").strip()
        if choice == "0":
            break
        if choice not in {"1", "2", "3"}:
            print("Invalid selection.")
            continue

        workflow_map = {"1": "img2img", "2": "inpainting", "3": "outpainting"}
        workflow = workflow_map[choice]
        print(f"\n--- {workflow} ---")
        run_repo_python("core/scripts/list_inputs.py", check=False)
        run_repo_python(
            "core/scripts/validate_capabilities.py",
            "--capability",
            workflow,
            "--summary",
            check=False,
        )
        comfy_url = RUNTIME.get("urls", {}).get("comfyui")
        if comfy_url:
            print(f"ComfyUI URL: {comfy_url}")
        else:
            print("ComfyUI URL: launch ComfyUI first (control panel option 1).")
        repo = resolve_repo_root_for_launch()
        print(f"Canonical workflow: {repo / f'workflows/base/{workflow}/workflow.json'}")
        print(f"Prepared workflows dir: {RUNTIME_WORKFLOWS_DIR}")
        print(f"Drive inputs: {DRIVE_INPUTS_DIR}")
        print("Implementation readiness is separate from execution input selection.")
        if workflow == "inpainting":
            check = run_repo_python("core/scripts/verify_models.py", "--require-inpainting", check=False)
            if check.returncode != 0:
                print("\nInpainting requires dedicated checkpoint:")
                print("  /content/drive/MyDrive/AI_Studio/models/shared/checkpoints/512-inpainting-ema.safetensors")
                print("  sd15.safetensors is not the inpainting checkpoint.")
                print("  Place the checkpoint in Drive, then retry inpainting.")
                continue

        input_path = input("Source image path (Enter to skip preparation): ").strip()
        if not input_path:
            print("Skipped preparation. Load the canonical workflow and set LoadImage manually.")
            continue

        args = ["--workflow", workflow, "--input", input_path]
        if workflow == "inpainting":
            mask_path = input("Mask image path: ").strip()
            if mask_path:
                args.extend(["--mask", mask_path])
        elif workflow == "outpainting":
            left = input("Expand left pixels [0]: ").strip() or "0"
            right = input("Expand right pixels [0]: ").strip() or "0"
            top = input("Expand top pixels [0]: ").strip() or "0"
            bottom = input("Expand bottom pixels [0]: ").strip() or "0"
            args.extend(["--left", left, "--right", right, "--top", top, "--bottom", bottom])
        run_repo_python("core/scripts/prepare_workflow.py", *args, check=False)
        print("Load the prepared workflow JSON in ComfyUI (Workflow → Load).")


def launch_mode(mode="safe"):
    studio_launch(mode)

print("Launchers ready.")


Launchers ready.


In [11]:
# ============================================================
# Cell 11 — Control Panel Menu
# ============================================================

def workspace_menu():
    while True:
        print("\n=== Workspace / Projects ===")
        run_repo_python("core/scripts/set_active_project.py", check=False)
        print("1. List projects")
        print("2. Create project")
        print("3. Switch active project")
        print("4. Show active project")
        print("5. Deactivate active project")
        print("6. Rename project")
        print("7. Archive project")
        print("8. Restore archived project")
        print("9. Delete project")
        print("10. Project statistics")
        print("11. Workflow catalog")
        print("12. Recent generations")
        print("13. Search generations")
        print("14. Generations menu")
        print("0. Back")
        choice = input("Select: ").strip()
        if choice == "0":
            break
        if choice == "1":
            run_repo_python("core/scripts/list_projects.py", check=False)
        elif choice == "2":
            name = input("Project display name: ").strip()
            if not name:
                continue
            description = input("Description (optional): ").strip()
            activate = input("Activate now? [y/N]: ").strip().lower() == "y"
            args = ["--name", name]
            if description:
                args.extend(["--description", description])
            if activate:
                args.append("--set-active")
            run_repo_python("core/scripts/create_project.py", *args, check=False)
        elif choice == "3":
            slug = input("Project slug or id to activate: ").strip()
            if slug:
                run_repo_python("core/scripts/set_active_project.py", "--slug", slug, check=False)
        elif choice == "4":
            run_repo_python("core/scripts/set_active_project.py", check=False)
        elif choice == "5":
            run_repo_python("core/scripts/deactivate_project.py", check=False)
        elif choice == "6":
            project = input("Project slug or id: ").strip()
            name = input("New display name (Enter to keep): ").strip()
            new_slug = input("New slug (Enter to keep folder): ").strip()
            if not project:
                continue
            args = ["--project", project]
            if name:
                args.extend(["--name", name])
            if new_slug:
                args.extend(["--new-slug", new_slug])
            if name or new_slug:
                run_repo_python("core/scripts/rename_project.py", *args, check=False)
        elif choice == "7":
            project = input("Project slug or id to archive: ").strip()
            if not project:
                continue
            print(f"Archiving {project} will deactivate it if it is active.")
            print("Existing assets will remain in Drive.")
            print("Future outputs will not be mirrored to this project.")
            confirm = input("Type YES to archive: ").strip()
            if confirm != "YES":
                print("Archive cancelled.")
                continue
            run_repo_python(
                "core/scripts/archive_project.py",
                "--project",
                project,
                "--yes",
                check=False,
            )
        elif choice == "8":
            project = input("Archived project slug or id: ").strip()
            if not project:
                continue
            activate = input("Activate after restore? [y/N]: ").strip().lower() == "y"
            args = ["--project", project]
            if activate:
                args.append("--set-active")
            run_repo_python("core/scripts/restore_project.py", *args, check=False)
        elif choice == "9":
            project = input("Project slug or id to delete: ").strip()
            if not project:
                continue
            print(f"Delete project: {project}")
            print("This permanently removes the managed project folder under AI_Studio/projects/.")
            print("Canonical files in AI_Studio/outputs/ will NOT be deleted.")
            dry = input("Run delete dry-run first? [Y/n]: ").strip().lower()
            if dry != "n":
                run_repo_python(
                    "core/scripts/delete_project.py",
                    "--project",
                    project,
                    "--dry-run",
                    check=False,
                )
            confirm = input("Type the exact project slug to confirm deletion: ").strip()
            if not confirm:
                print("Delete cancelled.")
                continue
            run_repo_python(
                "core/scripts/delete_project.py",
                "--project",
                project,
                "--confirm-slug",
                confirm,
                check=False,
            )
        elif choice == "10":
            project = input("Project slug or id: ").strip()
            if project:
                run_repo_python("core/scripts/project_statistics.py", "--project", project, check=False)
        elif choice == "11":
            run_repo_python("core/scripts/workflow_catalog.py", "--summary", check=False)
        elif choice == "12":
            run_repo_python("core/scripts/list_generations.py", check=False)
        elif choice == "13":
            args = []
            project = input("Filter project (optional): ").strip()
            capability = input("Filter capability (optional): ").strip()
            prompt = input("Prompt contains (optional): ").strip()
            date_from = input("Date from YYYY-MM-DD (optional): ").strip()
            date_to = input("Date to YYYY-MM-DD (optional): ").strip()
            if project:
                args.extend(["--project", project])
            if capability:
                args.extend(["--capability", capability])
            if prompt:
                args.extend(["--prompt-contains", prompt])
            if date_from:
                args.extend(["--date-from", date_from])
            if date_to:
                args.extend(["--date-to", date_to])
            run_repo_python("core/scripts/list_generations.py", *args, check=False)
        elif choice == "14":
            generations_menu()
        else:
            print("Invalid selection.")


def generations_menu():
    while True:
        print("\n=== Generations ===")
        print("1. Recent generations")
        print("2. Search generations")
        print("3. Show generation")
        print("4. Export generation")
        print("5. Validate generation snapshot")
        print("0. Back")
        choice = input("Select: ").strip()
        if choice == "0":
            break
        if choice == "1":
            run_repo_python("core/scripts/list_generations.py", check=False)
        elif choice == "2":
            args = []
            generation_id = input("Generation ID (optional): ").strip()
            project = input("Filter project (optional): ").strip()
            capability = input("Filter capability (optional): ").strip()
            prompt = input("Prompt contains (optional): ").strip()
            seed = input("Filter seed (optional): ").strip()
            snapshot = input("Snapshot status (optional): ").strip()
            if generation_id:
                args.extend(["--generation-id", generation_id])
            if project:
                args.extend(["--project", project])
            if capability:
                args.extend(["--capability", capability])
            if prompt:
                args.extend(["--prompt-contains", prompt])
            if seed:
                args.extend(["--seed", seed])
            if snapshot:
                args.extend(["--snapshot-status", snapshot])
            run_repo_python("core/scripts/list_generations.py", *args, check=False)
        elif choice == "3":
            generation_id = input("Generation ID: ").strip()
            if generation_id:
                run_repo_python(
                    "core/scripts/generation_info.py",
                    "--generation-id",
                    generation_id,
                    check=False,
                )
        elif choice == "4":
            generation_id = input("Generation ID to export: ").strip()
            if generation_id:
                run_repo_python(
                    "core/scripts/export_generation.py",
                    "--generation-id",
                    generation_id,
                    check=False,
                )
        elif choice == "5":
            generation_id = input("Generation ID to validate (Enter for all): ").strip()
            if generation_id:
                run_repo_python(
                    "core/scripts/validate_generation_snapshot.py",
                    "--generation-id",
                    generation_id,
                    check=False,
                )
            else:
                run_repo_python(
                    "core/scripts/validate_generation_snapshot.py",
                    "--all",
                    "--summary",
                    check=False,
                )
        else:
            print("Invalid selection.")


def report_output_autosync_status():
    """Read-only Package 4.5.2 current-runtime ownership and health status."""
    script = REPO_ROOT / "core" / "scripts" / "run_output_watcher.py"
    if not script.is_file():
        print(f"Watcher script missing: {script}")
        return
    try:
        completed = subprocess.run(
            [sys.executable, str(script), "--status"],
            cwd=str(REPO_ROOT),
            capture_output=True,
            text=True,
            timeout=60,
            check=False,
        )
    except Exception as exc:
        print(f"Unable to query output watcher status: {exc}")
        return
    text = (completed.stdout or "").strip() or (completed.stderr or "").strip()
    print(text or "(no watcher status)")
    report_script = REPO_ROOT / "core" / "scripts" / "runtime_report.py"
    if report_script.is_file():
        try:
            completed = subprocess.run(
                [sys.executable, str(report_script), "--summary"],
                cwd=str(REPO_ROOT),
                capture_output=True,
                text=True,
                timeout=120,
                check=False,
            )
            summary = (completed.stdout or "").strip()
            if summary:
                print(summary.splitlines()[-1])
        except Exception as exc:
            print(f"runtime_report summary skipped: {exc}")


def control_panel():
    while True:
        print("\n=== AI Studio Control Panel ===")
        print("1. Launch (Safe / Minimal / Full)")
        print("2. Model Manager")
        print("3. Node Manager")
        print("4. Backup")
        print("5. Restore")
        print("6. Full Reset")
        print("7. Image Editing")
        print("8. Output Watcher Status")
        print("9. Workspace / Projects")
        print("0. Exit")

        choice = input("Select: ").strip()

        try:
            if choice == "1":
                print("\nLaunch modes:")
                print("  safe    — ComfyUI only, no custom nodes, SD1.5 check")
                print("  minimal — ComfyUI + SD1.5 check + base txt2img workflow guidance")
                print("  full    — ComfyUI + stable custom nodes + SD1.5 check")
                mode = input("Mode [safe/minimal/full] (default: minimal): ").strip().lower()
                if not mode:
                    mode = "minimal"
                launch_mode(mode)
            elif choice == "2":
                model_manager_menu()
            elif choice == "3":
                node_manager_menu()
            elif choice == "4":
                backup_restore_menu()
            elif choice == "5":
                print(list_backups())
                name = input("Backup filename to restore: ").strip()
                restore_backup(name)
            elif choice == "6":
                confirm = input("Type RESET to confirm: ").strip()
                if confirm == "RESET":
                    full_reset_runtime_only()
                else:
                    print("Reset cancelled.")
            elif choice == "7":
                image_editing_menu()
            elif choice == "8":
                report_output_autosync_status()
            elif choice == "9":
                workspace_menu()
            elif choice == "0":
                print("Exiting control panel.")
                break
            else:
                print("Invalid selection.")
        except Exception as e:
            print(f"ERROR: {e}")

print("Control panel ready.")
print("Run control_panel() to start.")


Control panel ready.
Run control_panel() to start.


In [ ]:
control_panel()


=== AI Studio Control Panel ===
1. Launch (Safe / Minimal / Full)
2. Model Manager
3. Node Manager
4. Backup
5. Restore
6. Full Reset
0. Exit
Select: 1
Available modes: safe, minimal, full
Mode: minimal
$ apt-get update -qq
$ apt-get install -y -qq git git-lfs aria2 rsync
$ /usr/bin/python3 -m pip install -q requests packaging huggingface_hub safetensors
Installing ComfyUI in minimal mode...
$ git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI
$ /usr/bin/python3 -m pip install -q -r /content/ComfyUI/requirements.txt
Linked: /content/ComfyUI/models -> /content/drive/MyDrive/AI_Studio/models/shared
ComfyUI install complete.

=== sd15 ===
Already exists, skipping: sd15.safetensors
OK: /content/drive/MyDrive/AI_Studio/models/shared/checkpoints/sd15.safetensors (3.97 GB)
Mode install finished: minimal
Launching ComfyUI...
ComfyUI URL: https://8188-gpu-l4-s-kkb-ass1a1-1ke7lxis7yuqa-a.asia-southeast1-1.prod.colab.dev


Watchdog started for comfyui

=== AI Studio Control Panel ===
1. Launch (Safe / Minimal / Full)
2. Model Manager
3. Node Manager
4. Backup
5. Restore
6. Full Reset
0. Exit
